# RLHF와 DPO 선호최적화 실습 - 추가 실습 코드: DPO Loss 계산 시뮬레이션

- Tutorial ID: `mod-rlhf-dpo-lab`
- Tutorial: RLHF와 DPO 선호최적화 실습
- Section ID: `practice-dpo-loss`
- Section: 추가 실습 코드: DPO Loss 계산 시뮬레이션


In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: DPO Loss 계산 시뮬레이션
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   2) 선호쌍 chosen/rejected가 loss와 policy update 신호로 바뀌는 흐름 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def log_softmax(x):
    """수치 안정적인 log softmax"""
    return x - np.log(np.sum(np.exp(x - np.max(x))))

def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """
    Direct Preference Optimization (DPO) Loss
    
    DPO는 reward model 없이 직접 선호도를 학습합니다.
    
    L_DPO = -log(sigmoid(beta * (log(pi/pi_ref)_w - log(pi/pi_ref)_l)))
    
    Args:
        policy_chosen_logps: 현재 정책의 선호 응답 log 확률
        policy_rejected_logps: 현재 정책의 비선호 응답 log 확률
        ref_chosen_logps: 참조 정책의 선호 응답 log 확률
        ref_rejected_logps: 참조 정책의 비선호 응답 log 확률
        beta: KL divergence 페널티 계수
    """
    # Log ratio 계산
    chosen_ratio = policy_chosen_logps - ref_chosen_logps
    rejected_ratio = policy_rejected_logps - ref_rejected_logps
    
    # DPO 목표: 선호 응답의 ratio를 높이고, 비선호 응답의 ratio를 낮춤
        logits = beta * (chosen_ratio - rejected_ratio)
    
    # Negative log sigmoid loss
        loss = -np.log(1 / (1 + np.exp(-logits)))
    
    return loss, chosen_ratio, rejected_ratio

# 시뮬레이션
np.random.seed(42)

print("=" * 60)
print("DPO (Direct Preference Optimization) Loss")
print("=" * 60)

# 학습 전: 정책과 참조 정책이 비슷함
print("\\n1. 학습 초기 (policy ≈ ref):")
policy_chosen = -2.0
policy_rejected = -2.5
ref_chosen = -2.0
ref_rejected = -2.5

loss, c_ratio, r_ratio = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected)
print(f"   Chosen log(π/π_ref):   {c_ratio:.4f}")
print(f"   Rejected log(π/π_ref): {r_ratio:.4f}")
print(f"   DPO Loss: {loss:.4f}")

# 학습 후: 선호 응답 확률 상승, 비선호 응답 확률 하락
print("\\n2. 학습 후 (policy가 선호도 반영):")
policy_chosen = -1.5  # 선호 응답 확률 상승
policy_rejected = -3.0  # 비선호 응답 확률 하락

loss, c_ratio, r_ratio = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected)
print(f"   Chosen log(π/π_ref):   {c_ratio:.4f} ↑")
print(f"   Rejected log(π/π_ref): {r_ratio:.4f} ↓")
print(f"   DPO Loss: {loss:.4f} ↓")

# Beta의 영향
print("\\n3. Beta(β)의 영향:")
print("   β는 reference model에서 얼마나 벗어날 수 있는지 조절")
for beta in [0.05, 0.1, 0.5, 1.0]:
    loss, _, _ = dpo_loss(policy_chosen, policy_rejected, 
                          ref_chosen, ref_rejected, beta=beta)
    bar = "█" * int((1-loss) * 20)
    print(f"   β={beta:.2f}: Loss={loss:.4f} {bar}")

print("\\n💡 DPO의 핵심:")
print("   - Reward Model 없이 직접 선호도 학습")
print("   - RL(PPO) 대신 간단한 Cross-Entropy 형태")
print("   - β로 reference model과의 거리 조절")